# Liquidity Absorption EXT — the whole module, venue-native on Deribit, 2024-01-01 → latest

Pre-registration: `discussions/Liquidity_Absorption/024_claude_extended_deribit_full_period_preregistration.md`. Orchestration: `src/data_engineering/modeling/liquidity_absorption_extended.py`. Every stage engine is imported unchanged.

1. **Venue inputs** from the downloaded Deribit perpetual prints: aggTrades-schema tape, 1m bars in the kline schema, and the Deribit index 5m close as the VD18 input.
2. **A02** detector (frozen `Config`, period changed only) → **A02b** touch flow → **A03** FIXED / BE / TRAIL → **A03b** forward 1:1 (**primary**) → **A03c** reverse and maker → **A03d** VD18 risk → **A04** cohort → **A05a** context → **A06** Deribit derivatives context.
3. **Evaluation:**
   - Deribit fees (taker 5 bp, maker 0) and the frozen engine costs;
   - halves: H1 before 2025-04-01, H2 from then on; the in-sample Jul–Sep 2026 quarter is reported separately;
   - day-cluster bootstrap.
4. **Bridge:** Deribit-native signals vs the original 452 Binance signals in Jul–Sep 2026.

Stages skip when their output already exists (`LAEXT_FORCE=1` to recompute). `LAEXT_SYNTHETIC=1` for a smoke run. `LAEXT_SKIP_A06=1` to skip the options context.

In [1]:
from pathlib import Path
import json, os, sys, tempfile, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / 'src' / 'data_engineering').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / 'src'))
from src.data_engineering.modeling import liquidity_absorption_extended as ex
SYN = os.getenv('LAEXT_SYNTHETIC', '').strip() == '1'
FORCE = os.getenv('LAEXT_FORCE', '').strip() == '1'
SKIP_A06 = os.getenv('LAEXT_SKIP_A06', '').strip() == '1'
if SYN:
    TMP = Path(tempfile.mkdtemp(prefix='laext_')); HIST = TMP / 'history'; ROOT = TMP / 'venue'; OUT = TMP / 'EXT'
    days = [str(x.date()) for x in pd.date_range('2026-06-01', periods=14, freq='D')]
    ex.synthetic_history(HIST, days, seed=5); SPEC = ex.Spec(start='2026-06-04')
else:
    DATA = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
    HIST = DATA / 'history'; ROOT = DATA / 'la_ext_venue'; OUT = REPO / 'notebooks' / 'Liquidity_Absorption' / 'EXT'
    SPEC = ex.Spec(start=os.getenv('LAEXT_START', '2024-01-01'), end=os.getenv('LAEXT_END') or None)
OUT.mkdir(parents=True, exist_ok=True)
print(ex.VERSION, '| synthetic' if SYN else '', '| history', HIST, '| venue root', ROOT, '| out', OUT)
print(json.dumps(ex.registration(SPEC), indent=1))
def done(p): return (not FORCE) and Path(p).exists()
T = {}

LA-EXT-0.1.0  | history D:\MarketData\Deribit\history | venue root D:\MarketData\Deribit\la_ext_venue | out D:\Projects\deribit_market_data\notebooks\Liquidity_Absorption\EXT
{
 "version": "LA-EXT-0.1.0",
 "start": "2024-01-01",
 "end": null,
 "venue": "Deribit BTC-PERPETUAL / ETH-PERPETUAL prints",
 "in_sample_window": [
  "2026-07-01 00:00:00+00:00",
  "2026-10-01 00:00:00+00:00"
 ],
 "half_split": "2025-04-01 00:00:00+00:00",
 "deribit_fees_bp": {
  "taker": 5.0,
  "maker": 0.0
 },
 "boot_B": 2000,
 "primary": "A03b forward 1:1 taker (FIXED), closed trades, Deribit fees",
 "gate": "mean net bp per closed trade > 0 with one-sided 95% day-cluster bootstrap lower bound > 0, in BTC and ETH separately, and in both holdout halves (before / from 2025-04-01), holdout = all days outside 2026-07-01..2026-09-30"
}


## 1. Venue inputs (Deribit perpetual prints → tape, 1m bars, index 5m) — resumable per day

In [2]:
t0 = time.time()
prep_spec = ex.Spec(start=str((pd.Timestamp(SPEC.start) - pd.Timedelta(days=4)).date()), end=SPEC.end)   # A02 warm-up
COV = ex.prepare_venue_root(HIST, ROOT, prep_spec)
COV.to_csv(OUT / 'EXT_venue_coverage.csv', index=False)
for s in ex.SYMBOLS:
    b = pd.read_parquet(ROOT / f'{s}_um_1m.parquet', columns=['open_time', 'quote_volume'])
    print(s, 'bars', len(b), b.open_time.min(), '->', b.open_time.max(), '| zero-volume minutes', round(float((b.quote_volume == 0).mean()), 4))
print(f'{time.time()-t0:.0f}s')

  BTCUSDT 2024-01-01 (1/1023)
  BTCUSDT 2024-02-20 (51/1023)
  BTCUSDT 2024-04-10 (101/1023)
  BTCUSDT 2024-05-30 (151/1023)
  BTCUSDT 2024-07-19 (201/1023)
  BTCUSDT 2024-09-07 (251/1023)
  BTCUSDT 2024-10-27 (301/1023)
  BTCUSDT 2024-12-16 (351/1023)
  BTCUSDT 2025-02-04 (401/1023)
  BTCUSDT 2025-03-26 (451/1023)
  BTCUSDT 2025-05-15 (501/1023)
  BTCUSDT 2025-07-04 (551/1023)
  BTCUSDT 2025-08-23 (601/1023)
  BTCUSDT 2025-10-12 (651/1023)
  BTCUSDT 2025-12-01 (701/1023)
  BTCUSDT 2026-01-20 (751/1023)
  BTCUSDT 2026-03-11 (801/1023)
  BTCUSDT 2026-04-30 (851/1023)
  BTCUSDT 2026-06-19 (901/1023)
  BTCUSDT 2026-08-08 (951/1023)
  BTCUSDT 2026-09-27 (1001/1023)
  BTCUSDT 2026-10-19 (1023/1023)
  ETHUSDT 2024-01-01 (1/1023)
  ETHUSDT 2024-02-20 (51/1023)
  ETHUSDT 2024-04-10 (101/1023)
  ETHUSDT 2024-05-30 (151/1023)
  ETHUSDT 2024-07-19 (201/1023)
  ETHUSDT 2024-09-07 (251/1023)
  ETHUSDT 2024-10-27 (301/1023)
  ETHUSDT 2024-12-16 (351/1023)
  ETHUSDT 2025-02-04 (401/1023)
  ETHUSDT 20

## 2. A02 detector and A02b touch flow

In [3]:
t0 = time.time()
if not done(OUT / '02' / '02_causal_flow_features.csv'):
    print(json.dumps(ex.stage_a02(ROOT, OUT, SPEC)['summary'], indent=1, default=str))
if not done(OUT / '02b' / '02b_touch_features.csv'):
    print(ex.stage_a02b(ROOT, OUT))
c = pd.read_csv(OUT / '02' / '02_causal_flow_features.csv')
print('signals', len(c), c.groupby('symbol').size().to_dict(), f'| {time.time()-t0:.0f}s')

[
 {
  "symbol": "BTCUSDT",
  "minute_rows_including_warmup": 1473120,
  "price_only_candidates": 1204,
  "profile_present": 1151,
  "missing_trade_day_events": 0,
  "first_signal": "2024-01-02 21:48:00+00:00",
  "last_signal": "2026-10-19 17:31:00+00:00"
 },
 {
  "symbol": "ETHUSDT",
  "minute_rows_including_warmup": 1473120,
  "price_only_candidates": 1362,
  "profile_present": 1252,
  "missing_trade_day_events": 0,
  "first_signal": "2024-01-01 13:42:00+00:00",
  "last_signal": "2026-10-19 21:13:00+00:00"
 }
]


ValueError: Repeated A02 candidate identity

## 3. Executions: A03, A03b (primary), A03c, A03d

In [ ]:
t0 = time.time()
if not done(OUT / '03' / '03_execution_events.csv'): ex.stage_a03(ROOT, OUT); print('A03', f'{time.time()-t0:.0f}s')
if not done(OUT / '03b' / '03b_1to1_events.csv'): ex.stage_a03b(ROOT, OUT); print('A03b', f'{time.time()-t0:.0f}s')
if not done(OUT / '03c' / '03c_events.csv'): ex.stage_a03c(ROOT, OUT); print('A03c', f'{time.time()-t0:.0f}s')
if not done(OUT / '03d' / '03d_risk_execution.csv'): ex.stage_a03d(ROOT, OUT, REPO); print('A03d', f'{time.time()-t0:.0f}s')
b = pd.read_csv(OUT / '03b' / '03b_1to1_events.csv', low_memory=False)
display(b.groupby('symbol').reason.value_counts().unstack(fill_value=0))

## 4. Verdict (pre-registered gate on A03b forward 1:1 taker, Deribit fees) and all variants

In [ ]:
TAB, DEC = ex.evaluate(OUT)
TAB.to_csv(OUT / 'EXT_results_by_period.csv', index=False)
(OUT / 'EXT_decision.json').write_text(json.dumps(DEC, indent=1)); print(json.dumps(DEC, indent=1))
show = TAB[TAB.variant.eq('A03b_forward_1to1_taker')][['symbol', 'period', 'closed', 'target', 'stop', 'timeout', 'mean_gross_bp', 'gross_lo95', 'mean_net_bp', 'net_lo95']]
display(show.round(2))
display(TAB[TAB.period.eq('HOLDOUT') & TAB.symbol.eq('ALL')][['variant', 'closed', 'mean_gross_bp', 'gross_lo95', 'mean_net_bp', 'net_lo95']].round(2))

## 5. Bridge: Deribit-native vs original Binance signals, Jul–Sep 2026 (descriptive)

In [ ]:
if not SYN:
    BR = ex.bridge_vs_binance(OUT, REPO); BR.to_csv(OUT / 'EXT_bridge_jul_sep_2026.csv', index=False)
    m = BR[BR.matched]
    print('Binance signals with a Deribit counterpart:', round(BR.matched.mean(), 3), '| outcome agreement among matched:', round(m.agree.mean(), 3) if len(m) else None)
    display(BR.groupby(['symbol', 'matched']).size().unstack(fill_value=0))
    if len(m): display(pd.crosstab(m.binance_reason, m.deribit_reason))

## 6. Descriptive stages: A04 cohort, A05a context, A06 Deribit derivatives context

In [ ]:
t0 = time.time()
if not done(OUT / '04' / '04_primary_event_outcomes.csv'): ex.stage_a04(ROOT, OUT)
if not done(OUT / '05a' / '05a_phase_summary.csv'): print('A05a', ex.stage_a05a(ROOT, OUT))
print(f'{time.time()-t0:.0f}s')
if not SKIP_A06 and not done(OUT / '06' / '06_tests.csv'):
    from src.data_engineering.modeling import liquidity_absorption_stage_a06 as a6
    coh = a6.load_cohort(REPO, path=OUT / '04' / '04_primary_event_outcomes.csv', expected=None)
    cfg = a6.Config(n_controls=1, first_day=str(pd.to_datetime(coh.entry_ms.min(), unit='ms').date()),
                    last_day=str(pd.to_datetime(coh.entry_ms.max(), unit='ms').date()))
    ctl = a6.make_controls(coh, cfg); anch = pd.concat([coh, ctl], ignore_index=True)
    MIN6, COV6 = {}, []
    for cur in ('BTC', 'ETH'):
        a = anch[anch.currency.eq(cur)]
        d0 = pd.Timestamp(a.entry_ms.min() - 300 * 60_000, unit='ms', tz='UTC').normalize()
        d1 = pd.Timestamp(a.entry_ms.max() + 300 * 60_000, unit='ms', tz='UTC').normalize()
        dd = [str(x.date()) for x in pd.date_range(d0, d1, freq='D')]
        MIN6[cur], cv = a6.build_minutes(HIST, cur, dd, log=lambda c, d: print('  ', c, d, flush=True) if d.endswith('-01') else None)
        COV6.append(cv)
    o6 = OUT / '06'; o6.mkdir(exist_ok=True)
    pd.concat(COV6).to_csv(o6 / '06_coverage.csv', index=False)
    P6 = a6.event_panel(anch, MIN6, cfg)
    del MIN6
    ret = coh.set_index('anchor_id').gross_return_bp
    PH6 = a6.phase_panel(P6)
    PH6['winloss'] = PH6.anchor_id.map(lambda x: ('WIN' if ret.get(x) > 0 else 'LOSS') if pd.notna(ret.get(x, np.nan)) else None)
    a6.median_curves(P6).to_csv(o6 / '06_median_curves.csv', index=False)
    PH6.to_csv(o6 / '06_phase_panel.csv', index=False)
    sig = PH6[PH6.kind.eq('SIGNAL') & PH6.group.isin(a6.OUTCOMES)].assign(group='SIGNAL')
    wl = PH6[PH6.winloss.notna()].assign(group=lambda x: x.winloss)
    T6 = pd.concat([a6.compare(PH6, 'TARGET', 'STOP', s) for s in ('all', 'near_target', 'far_target')] +
                   [a6.compare(wl, 'WIN', 'LOSS'), a6.compare(pd.concat([sig, PH6[PH6.group.eq('CONTROL')]]), 'SIGNAL', 'CONTROL')], ignore_index=True)
    T6.to_csv(o6 / '06_tests.csv', index=False)
    (o6 / '06_decision.json').write_text(json.dumps(a6.decision(T6, P6, pd.concat(COV6), cfg), indent=1, default=str))
    print('A06 done', P6.groupby('group').anchor_id.nunique().to_dict(), f'{time.time()-t0:.0f}s')
if (OUT / '06' / '06_tests.csv').exists():
    T6 = pd.read_csv(OUT / '06' / '06_tests.csv')
    display(T6[T6.pre_entry & (T6.q_pre <= 0.10)].sort_values('q_pre').head(25).round(4))

## 7. Equity (A03b forward 1:1, Deribit fees, 1 unit per trade, not compounded) and the signal-rate check

In [ ]:
b = pd.read_csv(OUT / '03b' / '03b_1to1_events.csv', low_memory=False)
b['net_deribit'] = ex.deribit_net(b); z = b[b.reason.isin(ex.CLOSED)].copy()
z['t'] = pd.to_datetime(z.exit_time_ms, unit='ms', utc=True); z = z.sort_values('t')
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
for s, g in z.groupby('symbol'):
    ax[0].plot(g.t, g.net_deribit.cumsum() / 100, label=f'{s} net'); ax[0].plot(g.t, g.gross_return_bp.cumsum() / 100, ls='--', label=f'{s} gross')
for a_ in ex.INSAMPLE: ax[0].axvline(a_, c='k', ls=':', lw=1)
ax[0].axhline(0, c='k', lw=.5); ax[0].legend(); ax[0].set_title('cumulative % of notional per trade (A03b, Deribit fees)')
sig = pd.read_csv(OUT / '02' / '02_causal_flow_features.csv'); sig['m'] = pd.to_datetime(sig.signal_time_utc, utc=True).dt.to_period('M')
sig.groupby(['m', 'symbol']).size().unstack().plot(ax=ax[1]); ax[1].set_title('signals per month')
plt.tight_layout(); plt.savefig(OUT / 'EXT_equity.png', dpi=110); plt.show()